# Авторское решение: линейная регрессия и цена квартиры

Это не единственный правильный путь. Блокнот показывает аккуратный ход: validation, несколько признаков, сравнение линейных моделей и финальный `submission.csv`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_DIR = Path("data")


In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

target = "price_mln"
id_col = "id"


In [ ]:
def add_features(df):
    df = df.copy()
    df["area_per_room"] = df["area_m2"] / df["rooms"]
    df["kitchen_share"] = df["kitchen_m2"] / df["area_m2"]
    df["relative_floor"] = df["floor"] / df["floors_total"]
    df["is_first_floor"] = (df["floor"] == 1).astype(int)
    df["is_last_floor"] = (df["floor"] == df["floors_total"]).astype(int)
    df["no_metro"] = df["metro_min"].isna().astype(int)
    df["log_views"] = np.log1p(df["views_30d"])
    df["distance_squared"] = df["distance_to_center_km"] ** 2
    df["area_x_center"] = df["area_m2"] / (df["distance_to_center_km"] + 1)
    df["age_x_distance"] = df["house_age"] * df["distance_to_center_km"]
    return df


train_fe = add_features(train)
test_fe = add_features(test)

features = [c for c in train_fe.columns if c not in [target, id_col]]
X = train_fe[features]
y = train_fe[target]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)


In [ ]:
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def make_preprocess(X_part):
    numeric_features = X_part.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X_part.select_dtypes(exclude=np.number).columns.tolist()
    return ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_features,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", make_ohe()),
                    ]
                ),
                categorical_features,
            ),
        ]
    )


def evaluate_model(name, estimator):
    model = Pipeline(
        steps=[
            ("preprocess", make_preprocess(X_train)),
            ("model", estimator),
        ]
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    return {
        "name": name,
        "MAE": mean_absolute_error(y_val, pred),
        "RMSE": np.sqrt(mean_squared_error(y_val, pred)),
        "R2": r2_score(y_val, pred),
        "model": model,
    }


In [ ]:
candidates = [
    ("LinearRegression", LinearRegression()),
    ("Ridge alpha=0.3", Ridge(alpha=0.3)),
    ("Ridge alpha=1", Ridge(alpha=1.0)),
    ("Ridge alpha=3", Ridge(alpha=3.0)),
    ("Ridge alpha=10", Ridge(alpha=10.0)),
    ("Lasso alpha=0.001", Lasso(alpha=0.001, max_iter=10000)),
    ("ElasticNet alpha=0.001", ElasticNet(alpha=0.001, l1_ratio=0.3, max_iter=10000)),
]

rows = []
models = {}
estimators_by_name = {name: estimator for name, estimator in candidates}
for name, estimator in candidates:
    result = evaluate_model(name, estimator)
    models[name] = result.pop("model")
    rows.append(result)

leaderboard = pd.DataFrame(rows).sort_values("MAE")
leaderboard


In [ ]:
best_name = leaderboard.iloc[0]["name"]
best_name


In [ ]:
final_model = Pipeline(
    steps=[
        ("preprocess", make_preprocess(X)),
        ("model", clone(estimators_by_name[best_name])),
    ]
)

final_model.fit(X, y)
test_pred = final_model.predict(test_fe[features])

submission = pd.DataFrame(
    {
        "id": test_fe[id_col],
        "price_mln": np.maximum(test_pred, 0).round(3),
    }
)

submission.to_csv("author_submission.csv", index=False)
submission.head()
